In [ ]:
# ============================================================
# FULL UPDATED EDA PIPELINE USING CORRECTED COLUMN TYPES
# ============================================================

# Install required libs if needed
#import sys
#!{sys.executable} -m pip install polars plotly seaborn matplotlib ydata-profiling --quiet

# Imports
import polars as pl
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
#from ydata_profiling import ProfileReport

# ============================================================
# 1. LOAD FINAL PARQUET
# ============================================================

df = pl.read_parquet("data.parquet")

# ============================================================
# 2. COLUMN GROUPING (BASED ON VERIFIED TYPES)
# ============================================================

categorical_cols = [
    "COMPANY", "TYPE", "NAME",
    "BUCKET", "BASE", "GROUP", "UN",
    "OIL_RIG"
]

# SERVICE order sometimes float/int — keep numeric
numeric_cols = [
    "ID", "SERVICE",
    "ARQ_QTD", "ARQ_TAM",
    "FALTA_QTD", "FALTA_TAM",
    "FIELD", "DEPTH"
]

date_cols = ["OLDEST", "NEWEST", "DATE"]

# ============================================================
# 3. ENSURE DATE COLUMNS ARE PARSED
# ============================================================

for col in date_cols:
    df = df.with_columns(pl.col(col).cast(pl.Date))

# ============================================================
# 4. ENCODE CATEGORICAL COLUMNS
# ============================================================

df = df.with_columns([
    pl.col(col).cast(pl.Categorical) for col in categorical_cols
])

df_encoded = df.with_columns([
    pl.col(col).to_physical().alias(col + "_ENC") for col in categorical_cols
])

# ============================================================
# 5. TEXT EDA
# ============================================================

print("\n=== BASIC INFORMATION ===")
print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)

print("\n=== NUMERIC SUMMARY ===")
print(df.select(numeric_cols).describe())

print("\n=== NULL COUNTS ===")
print(df.null_count())

print("\n=== TOP 10 CATEGORIES ===")
for col in categorical_cols:
    print(f"\nTop 10 for {col}:")
    print(
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
    )

# ============================================================
# 6. CORRELATION MATRIX (Numeric + Encoded Categorical)
# ============================================================

numeric_for_corr = numeric_cols + [c + "_ENC" for c in categorical_cols]
pdf_corr = df_encoded.select(numeric_for_corr).to_pandas()
corr_matrix = pdf_corr.corr()

print("\nCorrelation matrix:")
print(corr_matrix)

# ============================================================
# 7. VISUAL EDA
# ============================================================

pdf = df_encoded.to_pandas()

# Histograms
for col in numeric_cols:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()

# Boxplots
for col in numeric_cols:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

# Correlation Heatmap
plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Heatmap (with Encoded Categories)")
plt.show()

# Category frequencies
for col in categorical_cols:
    top10 = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
          .to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10 {col}")
    fig.show()

# ============================================================
# 8. PROFILING REPORT
# ============================================================

#profile = ProfileReport(pdf, title="Dataset Profiling Report", explorative=True)
#profile.to_file("profiling_report.html")

print("\n✔ EDA Complete — profiling_report.html generated.")

In [5]:
!ls

data-01.parquet  IA-Submarina-OnBoarding
